# Tutorial 02 — OpenAI Runtime Adapter

This notebook starts from a real agent generated by **OpenAI Agent Builder** and shows
exactly what eXo-brain adds on top — using the **delegating wrapper** pattern behind the
provider-neutral `RuntimeAdapter` contract.

**Teaching vs production:** Act 2 defines an inline `OpenAIAgentsSDKAdapter` class so you
see the integration seam in one notebook. The shipped wheel is **`exo-adapter-openai`**
(`OpenAIAgentsRuntimeAdapter`, re-exported from `src/runtime/openai_agents_runtime`).
Verify PyPI provenance in `check_03_runtime_adapter.ipynb`.

**The story in three acts:**
1. **Before eXo-brain** — the agent runs but the tool body is `pass`, so nothing actually executes
2. **The adapter** — wrap the SDK behind the provider-neutral `RuntimeAdapter` contract
3. **After eXo-brain** — the same agent, same model, same tool schema — but now the tool runs
   deterministically with policy enforcement, risk gating, and a structured **`ToolResult`**
   envelope per call (persisted audit sinks are exercised in **Tutorial 04**, not here)

**Cells marked `[REQUIRES API KEY]` need `OPENAI_API_KEY` in your environment.**
All other cells run without credentials — including the policy enforcement demo.

In [1]:
import sys, pathlib, os, asyncio

# ── path setup ────────────────────────────────────────────────────────────────
_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
# ── load .env ─────────────────────────────────────────────────────────────────
_env = _root / ".env"
if _env.exists():
    from dotenv import load_dotenv
    load_dotenv(_env, override=False)
    print(f"✓ .env loaded from {_env}")
else:
    print(f"ℹ no .env at {_env}")


import importlib
import importlib.util

_ADAPTER_WHEELS = (
    ("exo-brain-core-contracts", "exo_brain_core_contracts"),
    ("exo-brain-adapter-sdk", "exo_brain_adapter_sdk"),
    ("exo-adapter-echo", "exo_adapter_echo"),
    ("exo-adapter-openai", "exo_adapter_openai"),
)


def _print_adapter_wheels() -> None:
    for dist, module_name in _ADAPTER_WHEELS:
        if importlib.util.find_spec(module_name) is None:
            print(f"warn: {dist} not installed — pip install -r requirements.txt")
            continue
        mod = importlib.import_module(module_name)
        mod_file = (mod.__file__ or "").replace("\\", "/")
        if "site-packages" not in mod_file and "dist-packages" not in mod_file:
            raise RuntimeError(f"{dist} must be a PyPI wheel in site-packages, got {mod.__file__}")
        if "/eXo_adapters/" in mod_file:
            raise RuntimeError(
                f"{dist} must not load from eXo_adapters checkout — "
                f"pip install -r requirements.txt: {mod.__file__}"
            )
        print(f"{dist}:", mod.__file__)


_print_adapter_wheels()

from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter

assert OpenAIAgentsRuntimeAdapter.__module__.startswith("exo_adapter_openai."), (
    "OpenAIAgentsRuntimeAdapter must come from exo-adapter-openai (PyPI); "
    "reinstall: pip install -r requirements.txt"
)

# ── framework imports ─────────────────────────────────────────────────────────
from src.runtime.runtime_adapter import RuntimeAdapter, SessionHandle
from src.runtime.capability_map import ProviderCapabilityMap, HealthStatus, HealthState, SecurityTier
from src.schemas.events import RuntimeEvent, RuntimeEventType
from src.schemas.tool_io import RiskTier, ToolCallContext, ToolResult, ToolStatus
from src.core.orchestrator import Orchestrator
from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolDescriptor, ToolRegistry

# ── openai agents sdk ─────────────────────────────────────────────────────────
from agents import Agent, Runner, function_tool, ModelSettings, TResponseInputItem

_key_set = bool(os.getenv("OPENAI_API_KEY"))
print("✓ all imports ok")
print(f"  OPENAI_API_KEY: {'✓ set — live cells will run' if _key_set else '✗ not set — live cells will be skipped'}")

# ── Agent instructions (shared across cells) ──────────────────────────────────
CALC_INSTRUCTIONS = (
    "You are a helpful math assistant. "
    "You MUST use the calculate_result function for every arithmetic operation. "
    "Supported operations: add, subtract, multiply, divide. "
    "Always call the function first, then explain the reasoning, then state the conclusion."
)
print(f"  CALC_INSTRUCTIONS defined")

✓ .env loaded from /home/razvansavin/Projects/eXo-brain/.env
exo-brain-core-contracts: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_brain_core_contracts/__init__.py
exo-brain-adapter-sdk: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_brain_adapter_sdk/__init__.py
exo-adapter-echo: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_adapter_echo/__init__.py
exo-adapter-openai: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_adapter_openai/__init__.py
✓ all imports ok
  OPENAI_API_KEY: ✓ set — live cells will run
  CALC_INSTRUCTIONS defined


---
## Act 1 — The original agent (as generated by OpenAI Agent Builder)

This is the exact Python code exported from OpenAI Agent Builder.

The agent is a math assistant that must call `calculate_result` for every arithmetic
operation. The tool parameters match the JSON schema embedded in the instructions:
`operation` (enum: add / subtract / multiply / divide), `operand1`, `operand2`.

**The problem:** `calculate_result` body is `pass` — it returns `None`.
The model dutifully calls it, but nothing happens. The result the model
receives back is always `None`, so it guesses from its own weights.

In [2]:
# ── Exact code from OpenAI Agent Builder ─────────────────────────────────────

@function_tool
def calculate_result(operation: str, operand1: float, operand2: float):
    """Performs a basic arithmetic calculation and returns the exact result."""
    pass   # ← unimplemented — model gets None back

exo_openai_agent = Agent(
    name="exo-openai-agent",
    instructions=CALC_INSTRUCTIONS,
    model="gpt-4o-mini",
    tools=[calculate_result],
    model_settings=ModelSettings(
        temperature=1,
        top_p=1,
        parallel_tool_calls=True,
        max_tokens=2048,
        store=True,
    ),
)

print("✓ exo_openai_agent defined (original, unmodified)")
print(f"  tools : {[t.name for t in exo_openai_agent.tools]}")
print(f"  model : {exo_openai_agent.model}")

✓ exo_openai_agent defined (original, unmodified)
  tools : ['calculate_result']
  model : gpt-4o-mini


### [REQUIRES API KEY] Run original agent — observe the `None` problem

In [3]:
if not os.getenv("OPENAI_API_KEY"):
    print("⚠  OPENAI_API_KEY not set — skipping")
else:
    print("▶ original agent  (calculate_result body = pass)...")
    print("─" * 60)
    result = await Runner.run(exo_openai_agent, "What is 5 plus 7?")
    print(f"  final output: {result.final_output!r}")
    print("─" * 60)
    print()
    print("The model called calculate_result, got None back, and guessed the answer.")
    print("No audit trail. No policy check. No error if the tool returns wrong data.")
    print("This is what eXo-brain fixes.")

▶ original agent  (calculate_result body = pass)...
────────────────────────────────────────────────────────────
  final output: 'It seems there was an issue retrieving the calculation result for 5 plus 7. However, logically, when you add 5 and 7 together, you combine the two quantities.\n\nSo, \\( 5 + 7 = 12 \\).\n\nThe conclusion is that 5 plus 7 equals 12.'
────────────────────────────────────────────────────────────

The model called calculate_result, got None back, and guessed the answer.
No audit trail. No policy check. No error if the tool returns wrong data.
This is what eXo-brain fixes.


---
## Act 2 — The adapter: wrapping the SDK behind the RuntimeAdapter contract

The `OpenAIAgentsSDKAdapter` uses the **delegating wrapper** pattern:

1. `sdk_tools` are `@function_tool` objects whose **bodies call eXo-brain's executor** directly
2. `Runner.run()` handles the full agentic loop — model emits a tool call, SDK calls the function, function delegates to eXo-brain, real result returned, model continues
3. The loop completes: model receives the actual result and produces a correct final answer

Why not stream interception? If the adapter intercepts a `tool_call_item` event and returns early, the SDK stream is abandoned — the model never receives the result and the loop can never complete.

```
model emits tool call
       │
SDK calls @function_tool body
       │
       └── ToolCallContext built from typed args
              │
       DeterministicToolExecutor.execute(call)
              │
              ├── PolicyMiddleware.before_tool_call()  risk=? → ALLOW/DENY
              ├── ModeSelector → DETERMINISTIC
              └── Real handler: _calculate_result(operation, operand1, operand2)
                     │
              ToolResult returned → body returns value to SDK
                     │
              SDK feeds result back to model ← full loop
                     │
              Model generates correct final answer ✓
```

In [4]:
import uuid
from typing import Any, AsyncIterator


class OpenAIAgentsSDKAdapter(RuntimeAdapter):
    """
    Wraps the OpenAI Agents SDK behind the provider-neutral RuntimeAdapter contract.

    Uses the 'delegating wrapper' pattern:
    - sdk_tools must be @function_tool objects whose BODIES call eXo-brain's executor.
    - Runner.run() handles the full agentic loop (model → tool → result → model → answer).
    - The adapter does NOT intercept stream events; execution flows through tool bodies.
    - submit_tool_results() is a fallback stub, not used in the delegating path.
    """

    def __init__(
        self,
        tool_registry: ToolRegistry,
        sdk_tools: list | None = None,
        provider_id: str = "openai",
        default_model: str = "gpt-4o-mini",
    ) -> None:
        self._registry      = tool_registry
        self._sdk_tools     = sdk_tools or []
        self._provider_id   = provider_id
        self._default_model = default_model
        self._sessions: dict[str, dict] = {}

    async def start_session(self, session_id: str, metadata: dict | None = None) -> SessionHandle:
        self._sessions[session_id] = {"history": [], "metadata": metadata or {}}
        return SessionHandle(session_id=session_id, provider_id=self._provider_id, metadata=metadata or {})

    async def run_turn(
        self, session_id: str, user_input: str, context: dict[str, Any],
    ) -> AsyncIterator[RuntimeEvent]:
        run_id  = str(context.get("run_id",  f"run_{uuid.uuid4().hex[:8]}"))
        corr_id = str(context.get("correlation_id", run_id))
        model   = str(context.get("model",   self._default_model))

        agent = Agent(
            name=str(context.get("agent_id", "agent_default")),
            instructions=str(context.get("instructions", "You are a helpful assistant.")),
            tools=self._sdk_tools,   # bodies already delegate to eXo-brain
            model=model,
        )

        try:
            # Full agentic loop: SDK handles model ↔ tool round-trips.
            # Each tool call routes through the @function_tool body → eXo-brain executor.
            result = await Runner.run(agent, user_input)

            # Update conversation history for multi-turn support
            session = self._sessions.setdefault(session_id, {"history": []})
            session["history"] = list(result.to_input_list())

            final = result.final_output or ""
            if final:
                yield RuntimeEvent.output_delta(
                    session_id=session_id, run_id=run_id,
                    text=final, correlation_id=corr_id,
                )
            yield RuntimeEvent.run_complete(
                session_id=session_id, run_id=run_id,
                output={"status": "completed", "provider_id": self._provider_id},
                correlation_id=corr_id,
            )

        except Exception as exc:
            yield RuntimeEvent.error(
                session_id=session_id, run_id=run_id,
                code="RUNTIME_TURN_ERROR", message=str(exc), correlation_id=corr_id,
            )

    async def submit_tool_results(self, session_id, run_id, tool_results):
        # Delegating pattern: tools execute inline via their bodies.
        # submit_tool_results() is only reached when the Orchestrator intercepts
        # a TOOL_INTENT event (e.g. from the simulation adapter in the policy demo).
        yield RuntimeEvent.run_complete(
            session_id=session_id, run_id=run_id,
            output={"status": "completed", "tool_results_count": len(tool_results)},
            correlation_id=run_id,
        )

    def get_capabilities(self) -> ProviderCapabilityMap:
        return ProviderCapabilityMap(
            provider_id=self._provider_id,
            supports_agents_sdk_native=True, supports_openai_compatible_api=False,
            supports_streaming=True, supports_function_calling=True,
            supports_structured_output=True, supports_handoffs=True,
            reliability_score=5, security_tier=SecurityTier.MANAGED_VENDOR,
            recommended_runtime_mode="hybrid",
        )

    async def healthcheck(self) -> HealthStatus:
        key = os.getenv("OPENAI_API_KEY", "")
        return HealthStatus(
            state=HealthState.HEALTHY if key else HealthState.DOWN,
            reason="api-key-present" if key else "no-api-key",
        )


print("✓ OpenAIAgentsSDKAdapter defined (delegating wrapper pattern)")

✓ OpenAIAgentsSDKAdapter defined (delegating wrapper pattern)


---
## Act 3 — Wire `calculate_result` into eXo-brain

Three-part wiring:

| Part | What it does |
|---|---|
| `_calculate_result(...)` registered in `ToolRegistry` | Real implementation, run deterministically by eXo-brain |
| `DeterministicToolExecutor` + `PolicyMiddleware` | Policy checks, audit logging, error envelopes |
| `@function_tool calculate_result(...)` with delegating body | Gives the model the JSON schema; body calls executor and returns real result |

The `@function_tool` body is the **integration seam**: it builds a `ToolCallContext`, calls `executor.execute()`, and returns the actual value to the SDK. The SDK feeds it to the model. The model generates the correct final answer.

In [5]:
# ── Step 1: Real implementation (runs inside eXo-brain) ──────────────────────

def _calculate_result(operation: str, operand1: float, operand2: float) -> dict:
    """Real calculate_result logic — deterministic, policy-gated, audited."""
    if operation == "add":
        value = operand1 + operand2
    elif operation == "subtract":
        value = operand1 - operand2
    elif operation == "multiply":
        value = operand1 * operand2
    elif operation == "divide":
        if operand2 == 0:
            raise ValueError("division by zero is not allowed")
        value = operand1 / operand2
    else:
        raise ValueError(f"unknown operation: {operation!r}")
    return {"operation": operation, "operand1": operand1, "operand2": operand2, "result": value}


# ── Step 2: Policy + executor (must exist before @function_tool body is called) ──
policy   = DeterministicFirstPolicyMiddleware()
registry = ToolRegistry()
registry.register(ToolDescriptor(
    name="calculate_result",
    handler=_calculate_result,
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
))
executor = DeterministicToolExecutor(registry=registry, policy=policy)


# ── Step 3: Delegating @function_tool — schema for model, body calls eXo-brain ──
@function_tool
def calculate_result(operation: str, operand1: float, operand2: float):
    """Performs a basic arithmetic calculation and returns the exact result."""
    call = ToolCallContext(
        schema_version="1.0",
        call_id=str(uuid.uuid4()),
        session_id="sess_exo", run_id="run_sdk",
        job_id="job_sdk",      task_id="task_sdk",
        agent_id="exo-openai-agent", provider_id="openai",
        tool_name="calculate_result",
        arguments={"operation": operation, "operand1": operand1, "operand2": operand2},
        risk_tier=RiskTier.LOW,
        is_state_changing=False,
    )
    tool_result = executor.execute(call)
    if tool_result.status == ToolStatus.SUCCESS:
        payload = tool_result.result or {}
        val = payload.get("result", payload.get("value", payload))
        print(f"  [eXo-brain] calculate_result({operation}, {operand1}, {operand2}) → {val}")
        return val
    raise ValueError(f"{tool_result.error.code}: {tool_result.error.message}")


# ── Wire adapter + orchestrator ───────────────────────────────────────────────
adapter = OpenAIAgentsSDKAdapter(
    tool_registry=registry,
    sdk_tools=[calculate_result],   # model sees typed schema; body delegates to eXo-brain
)
orchestrator = Orchestrator(
    runtime_adapter=adapter,
    policy_middleware=policy,
    tool_executor=executor,
)

print("✓ eXo-brain wired with calculate_result (delegating wrapper)")
print(f"  registry tools : {registry.list_tools()}")
health = await adapter.healthcheck()
print(f"  adapter health : {health.state.value} ({health.reason})")

✓ eXo-brain wired with calculate_result (delegating wrapper)
  registry tools : ['calculate_result']
  adapter health : healthy (api-key-present)


### [REQUIRES API KEY] Same agent, same question — now with real execution

In [6]:
if not os.getenv("OPENAI_API_KEY"):
    print("⚠  OPENAI_API_KEY not set — skipping")
else:
    context = {
        "run_id":       "run_exo_1",
        "job_id":       "job_exo",
        "task_id":      "task_exo",
        "agent_id":     "exo-openai-agent",
        "instructions": CALC_INSTRUCTIONS,
        "model":        "gpt-4o-mini",
    }

    async def live_turn(prompt: str):
        print(f"user ▶ {prompt}")
        print("─" * 60)
        # eXo-brain execution prints come from inside the @function_tool body
        # (above the dashes), then adapter events appear below.
        events = []
        async for event in orchestrator.run_turn("sess_exo", prompt, context):
            events.append(event)
            etype = event.event_type
            if etype == RuntimeEventType.OUTPUT_DELTA:
                text = event.payload.get("text", "")
                if text:
                    print(f"  [OUTPUT_DELTA]  {text[:200]!r}{'...' if len(text) > 200 else ''}")
            elif etype == RuntimeEventType.RUN_COMPLETE:
                print(f"  [RUN_COMPLETE]  status={event.payload.get('status')}")
            elif etype == RuntimeEventType.ERROR:
                print(f"  [ERROR]         {event.payload}")
        print("─" * 60)
        return events

    print("Test 1 — addition")
    await live_turn("What is 5 plus 7?")
    print()
    print("Test 2 — multiplication")
    await live_turn("What is 8 multiplied by 9?")
    print()
    print("Test 3 — subtraction")
    await live_turn("What is 100 minus 37?")

Test 1 — addition
user ▶ What is 5 plus 7?
────────────────────────────────────────────────────────────
  [eXo-brain] calculate_result(add, 5.0, 7.0) → {'operation': 'add', 'operand1': 5.0, 'operand2': 7.0, 'result': 12.0}
  [OUTPUT_DELTA]  'The result of adding 5 and 7 is 12. \n\nTo arrive at this conclusion, we take the two numbers and perform the addition operation: \n\n\\[ 5 + 7 = 12 \\]\n\nThus, the final answer is 12.'
  [RUN_COMPLETE]  status=completed
────────────────────────────────────────────────────────────

Test 2 — multiplication
user ▶ What is 8 multiplied by 9?
────────────────────────────────────────────────────────────
  [eXo-brain] calculate_result(multiply, 8.0, 9.0) → {'operation': 'multiply', 'operand1': 8.0, 'operand2': 9.0, 'result': 72.0}
  [OUTPUT_DELTA]  'To find the product of 8 multiplied by 9, I calculated it as follows:\n\n\\[\n8 \\times 9 = 72\n\\]\n\nThus, the conclusion is that \\( 8 \\) multiplied by \\( 9 \\) equals \\( 72 \\).'
  [RUN_COMPLETE]  sta

### [REQUIRES API KEY] Division by zero — model behaviour vs tool contract

This cell is a **behaviour discussion**, not a formal proof in this notebook.
The `calculate_result` handler raises `ValueError("division by zero is not allowed")`
when wired through eXo-brain. Depending on model behaviour, you may see:

- the model answer from its own knowledge (undefined / infinity text), or
- a delegated tool path that returns a structured error envelope.

Treat this as a place to **observe** behaviour. For a deterministic test of
structured tool errors, see the audit / tool envelope tutorials and tests.

In [7]:
if not os.getenv("OPENAI_API_KEY"):
    print("⚠  OPENAI_API_KEY not set — skipping")
else:
    print("Test 4 — division by zero")
    await live_turn("What is 10 divided by 0?")
    print()
    print("Note: in this live path the model may answer from its own knowledge.")
    print("For strict, testable error envelopes, rely on deterministic tests / Tutorial 04.")

Test 4 — division by zero
user ▶ What is 10 divided by 0?
────────────────────────────────────────────────────────────
  [OUTPUT_DELTA]  "Dividing by zero is undefined in mathematics. When you try to divide a number by zero, it means you're trying to find how many times zero can fit into that number, but zero cannot multiply to make any"...
  [RUN_COMPLETE]  status=completed
────────────────────────────────────────────────────────────

Note: in this live path the model may answer from its own knowledge.
For strict, testable error envelopes, rely on deterministic tests / Tutorial 04.


---
## Policy demo — HIGH risk calculation (no API key needed)

This cell uses the simulation adapter to show policy middleware in action.
Changing `risk_tier` to `HIGH` forces the mode selector to choose DETERMINISTIC
even for a simple arithmetic tool.

In [8]:
# No API key needed — uses simulation path with planned_tool_call injection

from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter as SimAdapter

high_registry = ToolRegistry()
high_registry.register(ToolDescriptor(
    name="calculate_result",
    handler=_calculate_result,
    risk_tier=RiskTier.HIGH,        # ← HIGH is configured to run deterministically in this demo
    is_state_changing=False,
))

sim_policy = DeterministicFirstPolicyMiddleware()
sim_orc    = Orchestrator(
    runtime_adapter=SimAdapter(),
    policy_middleware=sim_policy,
    tool_executor=DeterministicToolExecutor(registry=high_registry, policy=sim_policy),
)

sim_context = {
    "run_id": "run_policy", "job_id": "j_policy",
    "task_id": "t_policy",  "agent_id": "a_policy",
    "planned_tool_call": {
        "call_id":            "tc_policy",
        "tool_name":          "calculate_result",
        "arguments":          {"operation": "multiply", "operand1": 8, "operand2": 9},
        "risk_tier":          RiskTier.HIGH.value,
        "is_state_changing":  False,
    },
}

async def policy_demo():
    events = []
    async for event in sim_orc.run_turn("sess_policy", "8 * 9", sim_context):
        events.append(event)
        if event.event_type == RuntimeEventType.OUTPUT_DELTA:
            print(f"  [OUTPUT_DELTA]  {event.payload.get('text','')!r}")
        elif event.event_type == RuntimeEventType.RUN_COMPLETE:
            print(f"  [RUN_COMPLETE]  results={event.payload.get('tool_results_count')}")
    return events

print("HIGH-risk calculate_result (operation=multiply, 8×9) through policy middleware...")
print("─" * 60)
await policy_demo()
print("─" * 60)
print()
print("✓ HIGH-risk tool executed via deterministic path in this demo")
print("  Result: 72  |  Structured ToolResult envelope | Planned tool call injected; executor+handler ran in Python")

HIGH-risk calculate_result (operation=multiply, 8×9) through policy middleware...
────────────────────────────────────────────────────────────
  [OUTPUT_DELTA]  "- calculate_result (tc_policy): success → {'operation': 'multiply', 'operand1': 8, 'operand2': 9, 'result': 72}"
  [RUN_COMPLETE]  results=1
────────────────────────────────────────────────────────────

✓ HIGH-risk tool executed via deterministic path in this demo
  Result: 72  |  Structured ToolResult envelope | Planned tool call injected; executor+handler ran in Python


---
## Summary

| | Original agent (Agent Builder) | With eXo-brain |
|---|---|---|
| `calculate_result` body | `pass` → model gets `None` | delegates to `executor.execute()` → real result |
| Agentic loop | broken — model never gets result | ✅ full loop: model → tool → result → model → answer |
| Model sees tool schema | ✅ same | ✅ same |
| Execution path | SDK calls handler → `None` | `@function_tool` body → `DeterministicToolExecutor` → `_calculate_result` |
| Policy check | ✗ | ✅ `DeterministicFirstPolicyMiddleware` (`before_tool_call`) on delegated path |
| Audit envelope | ✗ | ✅ structured `ToolResult` envelope per call (core audit sinks exercised elsewhere) |
| Division by zero | model may hallucinate | ⚠️ behaviour discussion only; see Tutorial 04 / tests for strict error proofs |
| Risk gating | ✗ | ✅ LOW / MEDIUM / HIGH / CRITICAL tiers configured; detailed behaviour covered in policy tutorials |
| Provider swap | ✗ hardcoded OpenAI | ✅ adapter contract supports swap; this notebook focuses on the OpenAI Agents path |

**The `@function_tool` body is the integration seam. Everything outside it is already provider-neutral.**

### Production path
- **Shipped adapter:** `exo-adapter-openai` → `OpenAIAgentsRuntimeAdapter` (same contract as the inline class above)
- **Smoke check:** `check_03_runtime_adapter.ipynb` (wheel probe + `planned_tool_call` intent path)
- **Deterministic reference adapter:** `exo-adapter-echo` (`EchoRuntimeAdapter`) — see `check_03` and `tests/packages/test_echo_adapter_conformance.py`

### Next steps
- **Multi-turn** — call `run_turn()` again; session history is preserved in `adapter._sessions`
- **More tools** — register in `ToolRegistry` + delegating `@function_tool` wrapper
- **Ollama / local model** — same `RuntimeAdapter` contract, different `run_turn()` backend
- **Background pipelines** — wrap turns inside `BackgroundRuntime` DAG nodes

## Notebook navigation

| If you want… | Open |
|---|---|
| Previous / next in learning path | See `notebooks/README.md` index |
| Fast module smoke after a code change | `check_01` … `check_04` |
| Ingress or tool boundary proofs | `edge_01`, `edge_02` |
| Local governance lab (no API key) | `tutorial_08_governed_execution_sandbox.ipynb` |
| Live governance contrasts (optional API key) | `tutorial_09_governed_execution_live.ipynb` |
| Evaluator time-boxed paths | `notebooks/EVALUATOR_GUIDE.md` |

**Regenerate notebooks:** edit this build script, then `python notebooks/build_tutorials.py` (do not hand-edit `.ipynb` JSON).